# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [6]:
# Escribe tu código aquí para limpiar y enriquecer los datos
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# 1) Carga de datos (train set del EDA)
train_path = Path("../data/interim/train_set.csv")
housing = pd.read_csv(train_path)

print(f"Dataset cargado desde: {train_path}")
print(f"Dimensión original: {housing.shape}")

# 2) Limpieza de calidad de datos
# Decisión de completitud: imputar total_bedrooms con la mediana
if "total_bedrooms" in housing.columns:
    median_bedrooms = housing["total_bedrooms"].median()
    housing["total_bedrooms"] = housing["total_bedrooms"].fillna(median_bedrooms)

# Decisión de consistencia: eliminar duplicados exactos (si existieran)
duplicates_before = housing.duplicated().sum()
housing = housing.drop_duplicates().reset_index(drop=True)

# Decisión de precisión: valores no válidos (<= 0) en columnas de conteo
# Se marcan como faltantes y luego se imputan con mediana
count_cols = ["total_rooms", "total_bedrooms", "population", "households"]
for col in count_cols:
    invalid_mask = housing[col] <= 0
    if invalid_mask.any():
        housing.loc[invalid_mask, col] = np.nan
    housing[col] = housing[col].fillna(housing[col].median())

# Decisión de sensibilidad: winsorización simple por IQR para reducir impacto de outliers
def iqr_clip(series: pd.Series) -> pd.Series:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower=lower, upper=upper)

for col in ["total_rooms", "total_bedrooms", "population", "households", "median_income", "housing_median_age"]:
    housing[col] = iqr_clip(housing[col])

# 3) Feature engineering (variables combinadas)
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]

# 4) Codificación categórica
# ocean_proximity es nominal (no hay orden natural), por eso usamos one-hot encoding
housing_encoded = pd.get_dummies(housing, columns=["ocean_proximity"], prefix="ocean", dtype=int)

# 5) Escalado de variables numéricas
# Estandarizamos solo features (no el target)
target_col = "median_house_value"
X = housing_encoded.drop(columns=[target_col])
y = housing_encoded[target_col].copy()

dummy_cols = [c for c in X.columns if c.startswith("ocean_")]
numeric_features = X.select_dtypes(include="number").columns.difference(dummy_cols)

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_features] = scaler.fit_transform(X[numeric_features])

housing_prepared = pd.concat([X_scaled, y], axis=1)

joblib.dump(
    {"scaler": scaler, "feature_columns": list(X_scaled.columns)},
    "../models/preprocessor.pkl"
)

print("\n=== RESUMEN DE DECISIONES ===")
print("- Completitud: total_bedrooms imputado con mediana.")
print(f"- Consistencia: duplicados eliminados = {duplicates_before}.")
print("- Precisión: conteos <= 0 tratados como inválidos e imputados por mediana.")
print("- Sensibilidad: outliers acotados con IQR clipping en columnas numéricas clave.")
print("- Codificación: one-hot encoding en ocean_proximity por ser variable nominal.")
print("- Escalado: StandardScaler aplicado a las features numéricas.")

print(f"\nDimensión final (preparada): {housing_prepared.shape}")
print(housing_prepared.head())


Dataset cargado desde: ..\data\interim\train_set.csv
Dimensión original: (16512, 10)

=== RESUMEN DE DECISIONES ===
- Completitud: total_bedrooms imputado con mediana.
- Consistencia: duplicados eliminados = 0.
- Precisión: conteos <= 0 tratados como inválidos e imputados por mediana.
- Sensibilidad: outliers acotados con IQR clipping en columnas numéricas clave.
- Codificación: one-hot encoding en ocean_proximity por ser variable nominal.
- Escalado: StandardScaler aplicado a las features numéricas.

Dimensión final (preparada): (16512, 17)
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0   1.172993 -1.350415            0.428537     2.326857        2.134704   
1   1.268028 -1.378536           -1.473509    -1.106540       -1.192821   
2  -1.352939  0.988349           -0.046974     2.326857        2.331267   
3  -1.127856  0.758691           -0.284730     1.126331        0.460412   
4   1.793222 -1.083261           -1.632013    -1.579594       -1.603496   

 